# 🚀 Notebook 4: Production Concerns — Fan-out, Rate Limiting, Circuit Breakers

Real notification services don't die from the happy path — they die from **one provider going down** or **one campaign flooding one channel**. This notebook shows the three patterns that keep production systems alive.

## 🛠️ Setup

```bash
cd 06-system-designs/notification-system
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Fan-out — one event, many channels

Product says: *"When an order ships, push a notification AND email a receipt AND SMS the
tracking number (if user opted in)."*

The **caller doesn't need to know about all three channels.** It sends one event; the
notification service fans it out based on user preferences.

In [1]:
# A tiny in-memory preferences store + fan-out dispatcher.
prefs = {
    1: {"push": True,  "email": True,  "sms": True},
    2: {"push": True,  "email": True,  "sms": False},   # opted out of SMS
    3: {"push": False, "email": True,  "sms": False},   # email only
}

def fanout(user_id, event, payload):
    user_prefs = prefs.get(user_id, {})
    jobs = []
    for channel, enabled in user_prefs.items():
        if enabled:
            jobs.append({"user": user_id, "channel": channel,
                         "event": event, "payload": payload,
                         "dedup_key": f"{event}-{payload['order_id']}-{channel}"})
    return jobs

for uid in (1, 2, 3):
    print(f"user {uid} →")
    for job in fanout(uid, "order_shipped", {"order_id": 77}):
        print("   ", job)


user 1 →
    {'user': 1, 'channel': 'push', 'event': 'order_shipped', 'payload': {'order_id': 77}, 'dedup_key': 'order_shipped-77-push'}
    {'user': 1, 'channel': 'email', 'event': 'order_shipped', 'payload': {'order_id': 77}, 'dedup_key': 'order_shipped-77-email'}
    {'user': 1, 'channel': 'sms', 'event': 'order_shipped', 'payload': {'order_id': 77}, 'dedup_key': 'order_shipped-77-sms'}
user 2 →
    {'user': 2, 'channel': 'push', 'event': 'order_shipped', 'payload': {'order_id': 77}, 'dedup_key': 'order_shipped-77-push'}
    {'user': 2, 'channel': 'email', 'event': 'order_shipped', 'payload': {'order_id': 77}, 'dedup_key': 'order_shipped-77-email'}
user 3 →
    {'user': 3, 'channel': 'email', 'event': 'order_shipped', 'payload': {'order_id': 77}, 'dedup_key': 'order_shipped-77-email'}


**Notice the dedup_key pattern**: `{event}-{id}-{channel}`. Same event fanning out to
3 channels produces 3 distinct keys — each channel is independently idempotent, so a
retry on just the SMS step won't re-push or re-email.

## 2. Per-provider rate limiting

Twilio caps you at, say, **100 SMS/sec**. APNs has its own limits. Exceeding them gets
you throttled — or banned. We use a **token bucket** per provider.

**Bad approach**: one global rate limit for the whole service. One slow provider would
drag everything down.

**Good approach**: one bucket *per provider*, sized to that provider's SLA.

In [2]:
import time

class TokenBucket:
    """Classic token bucket: `rate` tokens/sec, max `capacity` tokens in reserve."""
    def __init__(self, rate, capacity):
        self.rate, self.capacity = rate, capacity
        self.tokens = capacity
        self.last   = time.monotonic()

    def try_acquire(self, n=1):
        now = time.monotonic()
        # refill based on elapsed time
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= n:
            self.tokens -= n
            return True
        return False

# One bucket per provider, sized to the provider's limit.
buckets = {
    "twilio": TokenBucket(rate=100, capacity=100),    # 100 SMS/sec
    "apns":   TokenBucket(rate=1000, capacity=1000),  # 1k push/sec
    "smtp":   TokenBucket(rate=50,   capacity=50),
}

def send_via(provider, msg):
    if not buckets[provider].try_acquire():
        return "RATE_LIMITED — requeue with small delay"
    return f"sent via {provider}: {msg}"

# Burst 200 SMS — only ~100 will get through before we're rate-limited.
results = [send_via("twilio", f"sms-{i}") for i in range(200)]
ok = sum(1 for r in results if r.startswith("sent"))
limited = sum(1 for r in results if r.startswith("RATE"))
print(f"sent through:  {ok}")
print(f"rate-limited:  {limited}  (these get requeued with a small delay)")


sent through:  100
rate-limited:  100  (these get requeued with a small delay)


Rate-limited messages should be **requeued with a small delay** (e.g. 100ms), not
dropped. Combined with the retry logic from Notebook 3, the system smoothly drains any
burst that exceeds the provider's capacity.

## 3. Circuit breaker — fail fast when a provider is down

When Twilio is actually down, sending 10,000 retries per second just wastes CPU and
connections. A **circuit breaker** is a three-state switch:

| State | Behaviour |
|---|---|
| `CLOSED` | Normal. Requests flow through. Count failures. |
| `OPEN`   | Too many failures → short-circuit all calls for a cooldown period. |
| `HALF_OPEN` | After cooldown, let ONE probe request through. Success → back to `CLOSED`. Failure → back to `OPEN`. |

This protects the upstream **and** fails fast so retries don't pile up.

In [3]:
import time, random

class CircuitBreaker:
    def __init__(self, fail_threshold=5, cooldown_s=2.0):
        self.fail_threshold = fail_threshold
        self.cooldown_s     = cooldown_s
        self.state          = "CLOSED"
        self.fails          = 0
        self.opened_at      = 0.0

    def call(self, fn):
        # Maybe transition OPEN -> HALF_OPEN after cooldown
        if self.state == "OPEN" and time.monotonic() - self.opened_at >= self.cooldown_s:
            self.state = "HALF_OPEN"

        if self.state == "OPEN":
            return "SHORT_CIRCUIT"

        try:
            result = fn()
            # success
            self.fails = 0
            self.state = "CLOSED"
            return result
        except Exception:
            self.fails += 1
            if self.state == "HALF_OPEN" or self.fails >= self.fail_threshold:
                self.state = "OPEN"
                self.opened_at = time.monotonic()
            return "FAIL"

# Simulate a provider that is fully down for a while, then recovers.
down_until = time.monotonic() + 1.5           # down for 1.5s

def provider():
    if time.monotonic() < down_until:
        raise RuntimeError("provider down")
    return "OK"

cb = CircuitBreaker(fail_threshold=3, cooldown_s=0.5)

log = []
for i in range(25):
    log.append((round(time.monotonic() - (down_until - 1.5), 2),
                cb.state, cb.call(provider)))
    time.sleep(0.15)

for t, state, result in log:
    print(f"t={t:>4.2f}s  state_before={state:<10}  result={result}")


t=0.00s  state_before=CLOSED      result=FAIL
t=0.30s  state_before=CLOSED      result=FAIL
t=0.59s  state_before=CLOSED      result=FAIL
t=0.87s  state_before=OPEN        result=SHORT_CIRCUIT
t=1.17s  state_before=OPEN        result=FAIL
t=1.45s  state_before=OPEN        result=SHORT_CIRCUIT
t=1.75s  state_before=OPEN        result=OK
t=1.96s  state_before=CLOSED      result=OK
t=2.25s  state_before=CLOSED      result=OK
t=2.54s  state_before=CLOSED      result=OK
t=2.81s  state_before=CLOSED      result=OK
t=3.03s  state_before=CLOSED      result=OK
t=3.23s  state_before=CLOSED      result=OK
t=3.53s  state_before=CLOSED      result=OK
t=3.83s  state_before=CLOSED      result=OK
t=4.11s  state_before=CLOSED      result=OK
t=4.40s  state_before=CLOSED      result=OK
t=4.69s  state_before=CLOSED      result=OK
t=4.98s  state_before=CLOSED      result=OK
t=5.26s  state_before=CLOSED      result=OK
t=5.56s  state_before=CLOSED      result=OK
t=5.85s  state_before=CLOSED      result=OK
t=

Watch the output:

1. First failures → breaker trips `OPEN`.
2. Subsequent calls short-circuit (`SHORT_CIRCUIT`) for the cooldown — **zero load on the dead provider**.
3. After cooldown, one probe goes `HALF_OPEN`; if the provider has recovered, we snap back to `CLOSED`.

## 4. Observability — the metrics you **must** have

A notification service without metrics is flying blind. At minimum track:

| Metric | Why |
|---|---|
| `notifications_sent_total{channel, status}` | Throughput & success rate per channel |
| `provider_latency_seconds{provider}` histogram | Spot slow providers early |
| `retry_attempts_total{channel}` | Rising retries = provider health regression |
| `dlq_depth{channel}` | How many messages need human attention |
| `rate_limit_hits_total{provider}` | Are we pushing harder than the provider allows? |
| `circuit_breaker_state{provider}` (0/1/2) | One glance at provider health |

A simple dict-of-counters is enough to *learn* the idea:

In [4]:
from collections import defaultdict

class Metrics:
    def __init__(self):
        self.counters = defaultdict(int)
    def inc(self, name, **labels):
        key = (name, tuple(sorted(labels.items())))
        self.counters[key] += 1
    def dump(self):
        for (name, labels), v in sorted(self.counters.items()):
            lbl = ",".join(f"{k}={v}" for k,v in labels)
            print(f"  {name}{{{lbl}}} = {v}")

m = Metrics()
# Pretend a batch of sends happened:
m.inc("notifications_sent_total", channel="push",  status="ok")
m.inc("notifications_sent_total", channel="push",  status="ok")
m.inc("notifications_sent_total", channel="sms",   status="ok")
m.inc("notifications_sent_total", channel="sms",   status="fail")
m.inc("retry_attempts_total",    channel="sms")
m.inc("dlq_depth",               channel="sms")
m.dump()
print("\nIn production, swap this for Prometheus / OpenTelemetry.")


  dlq_depth{channel=sms} = 1
  notifications_sent_total{channel=push,status=ok} = 2
  notifications_sent_total{channel=sms,status=fail} = 1
  notifications_sent_total{channel=sms,status=ok} = 1
  retry_attempts_total{channel=sms} = 1

In production, swap this for Prometheus / OpenTelemetry.


## 5. What we left out (deliberately)

A real production system also handles:

- **Batching** outgoing sends for cost (one SES request with 50 recipients).
- **Template versioning & localization** (`order_shipped.en`, `order_shipped.fr`).
- **Delivery receipts / webhooks** from the provider back into `send_log`.
- **Abuse / spam detection** — cap notifications per user per day.
- **Cross-region failover** for high availability.
- **Compliance** — unsubscribe links, GDPR export/delete of notification history.

Each deserves its own deep dive — but the patterns in these four notebooks
(**queues → priorities → retries → idempotency → fan-out → rate limit → circuit breaker → metrics**)
are the load-bearing walls every notification service is built on.